# 第 4 章：多态与鸭子类型

> 本章目标：理解**多态（Polymorphism）**的思想，掌握 Python 特有的**鸭子类型（Duck Typing）**，对比静态语言的接口式多态，了解 `abc` 抽象基类与 `typing.Protocol` 两种规范化手段。

---

## 4.1 什么是多态？

**多态 = 同一个接口，不同的实现。** 调用方不关心对象的具体类型，只关心"你能不能做这件事"。

类比：你说"播放音乐"，手机、音箱、电脑各自用自己的方式完成——你不需要知道它们内部的差异。

```mermaid
flowchart TD
    U[调用方: 让所有动物 speak] --> A[Dog.speak → 汪汪]
    U --> B[Cat.speak → 喵喵]
    U --> C[Duck.speak → 嘎嘎]
    U --> D[Cow.speak → 哞哞]
    style U fill:#e1f5ff
```

多态的价值：**新增一个类型时，调用方代码一行都不用改**（开闭原则——对扩展开放，对修改关闭）。

In [1]:
class Dog:
    def speak(self):
        return "汪汪汪"

class Cat:
    def speak(self):
        return "喵喵喵"

class Duck:
    def speak(self):
        return "嘎嘎嘎"


def make_it_speak(animal):
    """调用方：不关心 animal 是什么类型，只要它有 speak 方法"""
    print(animal.speak())


for animal in [Dog(), Cat(), Duck()]:
    make_it_speak(animal)

# 新增一个类型，make_it_speak 完全不用改！
class Robot:
    def speak(self):
        return "哔哔哔（电子音）"

make_it_speak(Robot())

汪汪汪
喵喵喵
嘎嘎嘎
哔哔哔（电子音）


## 4.2 鸭子类型：Python 的灵魂

> *"If it walks like a duck and it quacks like a duck, then it must be a duck."*
> （如果它走起来像鸭子、叫起来像鸭子，那它就是鸭子。）

注意上面的代码：`Dog`、`Cat`、`Duck`、`Robot` **没有任何继承关系**！`make_it_speak` 也不检查类型——它只要求对象**有 `speak` 方法**。这就是鸭子类型：

**关注的不是"你是什么"，而是"你能做什么"。**

| 对比 | Java / C# | Python |
|------|-----------|--------|
| 多态前提 | 必须实现共同接口或继承共同父类 | 不需要，有同名方法即可 |
| 类型检查时机 | **编译期**（静态检查） | **运行期**（调用时找不到方法才报错） |
| 灵活性 | 低，结构严谨 | 高，"协议式"协作 |
| 风险 | 编译器兜底 | 传错对象运行时才炸 |

```mermaid
flowchart LR
    subgraph Java[静态语言: 名义类型]
        I[Speakable 接口] --> J1[Dog implements Speakable]
        I --> J2[Cat implements Speakable]
    end
    subgraph Py[Python: 结构类型]
        P1[Dog 有 speak] --- P2[Cat 有 speak]
        P2 --- P3[Robot 有 speak]
        P4["调用方: 有 speak 就行"] -.-> P1
        P4 -.-> P2
        P4 -.-> P3
    end
```

## 4.3 鸭子类型的经典应用：内置函数

Python 的内置函数大量依赖鸭子类型——`len()`、`for` 循环、`sum()` 都不检查类型，只检查"协议"：

| 操作 | 要求的协议（魔术方法） |
|------|----------------------|
| `len(x)` | `__len__` |
| `for i in x` | `__iter__` |
| `x[i]` | `__getitem__` |
| `x + y` | `__add__` |

只要你的类实现了对应魔术方法，就能被内置机制当作"那种东西"使用（第 5 章详细展开）。

In [2]:
class Fibonacci:
    """一个自定义"可迭代"对象——不是列表，但可以 for 循环！"""

    def __init__(self, n):
        self.n = n

    def __iter__(self):
        a, b = 0, 1
        for _ in range(self.n):
            yield a
            a, b = b, a + b


class WordCollection:
    """实现了 __len__，就可以用 len()"""

    def __init__(self, words):
        self.words = words

    def __len__(self):
        return len(self.words)


# Fibonacci 不是 list/tuple，但 for 循环照常用
print(list(Fibonacci(10)))

# WordCollection 不是内置容器，但 len() 照常用
wc = WordCollection(["Python", "鸭子类型", "真香"])
print(len(wc))

# sum、max 等也只看可迭代协议
print(sum(Fibonacci(10)))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
3
88


## 4.4 鸭子类型的代价与防御

鸭子类型的风险：**传错对象时，错误发生在运行期、且报错位置可能离出错原因很远**。

常见防御手段：

1. `hasattr` 检查能力；
2. `isinstance` 检查类型（配合抽象基类）；
3. 更 Pythonic 的方式：**直接调用，捕获异常**（EAFP 原则：Easier to Ask Forgiveness than Permission）。

In [3]:
def make_it_speak_lbyl(animal):
    """LBYL 风格：Look Before You Leap（先检查再跳）"""
    if hasattr(animal, "speak"):
        print(animal.speak())
    else:
        print("这个对象不会说话")

def make_it_speak_eafp(animal):
    """EAFP 风格：直接做，出错再补救（Python 更推荐）"""
    try:
        print(animal.speak())
    except AttributeError:
        print("这个对象不会说话")


class Cat:
    def speak(self):
        return "喵喵喵"

class Rock:
    pass   # 石头没有 speak

make_it_speak_lbyl(Cat())
make_it_speak_lbyl(Rock())
make_it_speak_eafp(Cat())
make_it_speak_eafp(Rock())

喵喵喵
这个对象不会说话
喵喵喵
这个对象不会说话


## 4.5 用抽象基类（ABC）显式约束多态

当团队变大、项目变复杂，纯鸭子类型可能太"自由"。`abc` 模块让你像 Java 接口一样**强制子类实现某些方法**（详细语法见第 6 章）：

- 没实现抽象方法的子类**无法实例化**；
- 类型意图清晰，IDE 能给出提示。

In [4]:
from abc import ABC, abstractmethod

class Shape(ABC):
    """抽象基类：定义"形状"的接口"""

    @abstractmethod
    def area(self):
        """子类必须实现面积计算"""

    def describe(self):     # 抽象类也可以有具体方法
        return f"面积是 {self.area():.2f}"


class Circle(Shape):
    def __init__(self, r):
        self.r = r

    def area(self):
        import math
        return math.pi * self.r ** 2


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


shapes = [Circle(5), Square(4)]
for s in shapes:
    print(s.describe())      # 多态：同一接口，不同实现

try:
    Shape()                  # ❌ 抽象类不能直接实例化
except TypeError as e:
    print(f"拦截: {e}")

面积是 78.54
面积是 16.00
拦截: Can't instantiate abstract class Shape without an implementation for abstract method 'area'


## 4.6 `typing.Protocol`：给鸭子类型加上静态检查

Python 3.8+ 的 `Protocol` 是两者之间的甜点：**保持鸭子类型的写法，但让 mypy 等工具能静态检查**：

- 类**不需要继承** Protocol，只要结构上满足就算实现；
- 这叫做**结构化类型（structural typing）**，类似 Go 的接口。

In [5]:
from typing import Protocol

class Speakable(Protocol):
    """协议：任何有 speak 方法的类都自动满足"""
    def speak(self) -> str: ...


class Dog:
    def speak(self) -> str:
        return "汪汪"

class Robot:
    def speak(self) -> str:
        return "哔哔"

class Rock:
    pass   # 没有 speak，不满足协议


def announce(speaker: Speakable) -> None:
    """类型注解告诉 mypy：这里需要 Speakable"""
    print(speaker.speak())


announce(Dog())      # ✅ mypy 检查通过
announce(Robot())    # ✅ mypy 检查通过（没继承也认）
# announce(Rock())   # ❌ mypy 会报错；运行时到 .speak 才 AttributeError

# 运行时也可以用 runtime_checkable 做 isinstance 检查
from typing import runtime_checkable

@runtime_checkable
class Quackable(Protocol):
    def quack(self) -> str: ...

print(isinstance(Dog(), Quackable))   # False：Dog 没有 quack

汪汪
哔哔
False


## 4.7 三种多态方案如何选择？

```mermaid
flowchart TD
    A[需要多态] --> B{项目规模 / 团队约束?}
    B -- 小项目 脚本 --> C[纯鸭子类型<br/>最 Pythonic]
    B -- 需要 mypy 静态检查 --> D[typing.Protocol<br/>结构不变 检查加强]
    B -- 框架 库 API 设计 --> E[abc 抽象基类<br/>强制实现 运行期拦截]
```

| 方案 | 检查时机 | 需要继承? | 典型场景 |
|------|---------|----------|---------|
| 鸭子类型 | 运行时（出错才报） | 否 | 日常业务代码 |
| Protocol | 静态（mypy） | 否 | 大项目、类型敏感代码 |
| ABC | 运行时（实例化时报） | 是 | 框架、插件体系 |

### 📝 动手练习

1. 定义 `PdfExporter`、`CsvExporter`、`JsonExporter` 三个类（**不继承**共同父类），各自有 `export(data)` 方法；写一个 `save_report(exporter, data)` 函数统一调用。
2. 给上题加上 `Protocol` 类型注解，装一个 mypy 试试 `mypy your_file.py`。
3. 思考题：`len()` 为什么不用 `isinstance(x, list)` 判断？这体现了什么设计哲学？

---
**下一章** 👉 `05_魔术方法与运算符重载.ipynb`：深入 `__str__`、`__add__`、`__getitem__` 等协议方法，让你的类像内置类型一样好用。